# Enhancing Obstacle Avoidance in Dynamic Window Approach via Dynamic Obstacle Behavior Prediction

**Bongsu Hahn** (Department of Mechanical and Design Engineering, Hongik University, Sejong 30016, Republic of Korea)

*Actuators* 2025, 14, 207. https://doi.org/10.3390/act14050207

---

## What this notebook gives you

- A complete implementation of the Enhanced DWA algorithm with dynamic obstacle behavior prediction
- A runnable demo on a simple 2D planning problem with dynamic obstacles
- Visualizations showing how the robot predicts obstacle motion and selects optimal velocities
- A modular, importable package you can use in your own robotics code

## Two ways to use this

1. **Run as-is**: Execute all cells to see the Enhanced DWA algorithm in action on the bundled demo problem.
2. **Import into your own code**: Use `from method import select_velocity, ...` to integrate the planner into your robotics simulation or real robot controller.

## What this notebook does NOT do

- This is a **smoke demo**, not a benchmark. It runs a single seed on a simple environment to demonstrate the algorithm works.
- No comparison against conventional DWA or other planners (those would require a benchmark suite).
- No real robot deployment (this is a simulation).

The paper's full evaluation uses 20 trials on structured and randomized obstacle environments; this notebook shows the core algorithm on a minimal example.

## Contents

- [0. Install dependencies (first run only)](#sec-0-install-dependencies-first-run-only)
- [1. Setup](#sec-1-setup)
- [2. Parameters](#sec-2-parameters)
  - [Optional: Scale up to paper-faithful values](#sec-optional-scale-up-to-paper-faithful-values)
- [3. The setup pieces](#sec-3-the-setup-pieces)
  - [3.1 Environment](#sec-31-environment)
  - [3.2 System dynamics](#sec-32-system-dynamics)
  - [3.3 Planning problem](#sec-33-planning-problem)
  - [3.4 Collision checking](#sec-34-collision-checking)
- [4. The Enhanced DWA method ⭐](#sec-4-the-enhanced-dwa-method)
  - [4.1 Intuition](#sec-41-intuition)
  - [4.2 Dynamic Obstacle Position Prediction ⭐](#sec-42-dynamic-obstacle-position-prediction)
  - [4.3 Modified Dynamic Window](#sec-43-modified-dynamic-window)
  - [4.4 Objective Function Evaluation](#sec-44-objective-function-evaluation)
  - [4.5 Robot State Update](#sec-45-robot-state-update)
  - [4.6 Putting it together](#sec-46-putting-it-together)
- [5. Running the planner end-to-end](#sec-5-running-the-planner-end-to-end)
  - [5.1 Solve the planning problem](#sec-51-solve-the-planning-problem)
  - [5.2 Visualize the trajectory](#sec-52-visualize-the-trajectory)
  - [5.3 Planner statistics](#sec-53-planner-statistics)
- [6. Use your own problem](#sec-6-use-your-own-problem)

<a id="sec-0-install-dependencies-first-run-only"></a>

## 0. Install dependencies (first run only)

Run this cell once to install the required dependencies. Subsequent runs can skip this step.

In [ ]:
# %pip install -r requirements.txt

<a id="sec-1-setup"></a>

## 1. Setup

In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt
import numpy as np
import torch

# Set master random seed for reproducibility
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

# Import the method package public API
from method import (
    select_velocity,
    compute_dynamic_window,
    evaluate_objective_function,
    predict_obstacle_position,
    update_robot_state,
    DifferentialDriveDynamics,
    CircularCollisionModel,
    precompute_motion_primitives,
    load_environment,
    load_problem,
)

<a id="sec-2-parameters"></a>

## 2. Parameters

The parameters below control the Enhanced DWA algorithm's behavior. Each parameter has a provenance:

| Source | Meaning |
|--------|---------|
| `paper` | Value explicitly stated in the paper |
| `system_default` | Value not in paper; chosen by the pipeline for smoke/demo scale |
| `system_inferred` | Value derived from other parameters (e.g., `d_safe = V_max / 4`) |

The table and dict below are auto-generated from `.pipeline/params.json`.

| Parameter | Variable from paper | Value from paper | Paper value | System value | Used? | Notes |
|---|:-:|:-:|---|---|:-:|---|
| `alpha_h` | ❌ | ❌ | — | 0.5 | ✅ | Value from the pluggable_component signature default (alpha_h=0.5). The... |
| `alpha_d` | ❌ | ❌ | — | 0.5 | ✅ | Value from the pluggable_component signature default (alpha_d=0.5). The... |
| `alpha_c` | ❌ | ❌ | — | 0.5 | ✅ | Value from the pluggable_component signature default (alpha_c=0.5). The... |
| `alpha_f` | ❌ | ❌ | — | 0.5 | ✅ | Value from the pluggable_component signature default (alpha_f=0.5). The... |
| `beta` | ❌ | ❌ | — | None | ✅ | Value from the pluggable_component signature default (beta=None). The... |

In [ ]:
params = {
    "alpha_h": {
        "value": 0.5,
        "source": 'spec_default',
        "reasoning": (
            "Value from the pluggable_component signature default (alpha_h=0.5). "
            "The signature default is a runtime convenience chosen by the analyzer; "
            "it may or may not match the paper's stated value. If the paper "
            "specifies a value, add a structured source to the spec and an entry to "
            "_PAPER_VALUE_SPEC_PATHS so this value is resolved against paper-truth."
        ),
    },
    "alpha_d": {
        "value": 0.5,
        "source": 'spec_default',
        "reasoning": (
            "Value from the pluggable_component signature default (alpha_d=0.5). "
            "The signature default is a runtime convenience chosen by the analyzer; "
            "it may or may not match the paper's stated value. If the paper "
            "specifies a value, add a structured source to the spec and an entry to "
            "_PAPER_VALUE_SPEC_PATHS so this value is resolved against paper-truth."
        ),
    },
    "alpha_c": {
        "value": 0.5,
        "source": 'spec_default',
        "reasoning": (
            "Value from the pluggable_component signature default (alpha_c=0.5). "
            "The signature default is a runtime convenience chosen by the analyzer; "
            "it may or may not match the paper's stated value. If the paper "
            "specifies a value, add a structured source to the spec and an entry to "
            "_PAPER_VALUE_SPEC_PATHS so this value is resolved against paper-truth."
        ),
    },
    "alpha_f": {
        "value": 0.5,
        "source": 'spec_default',
        "reasoning": (
            "Value from the pluggable_component signature default (alpha_f=0.5). "
            "The signature default is a runtime convenience chosen by the analyzer; "
            "it may or may not match the paper's stated value. If the paper "
            "specifies a value, add a structured source to the spec and an entry to "
            "_PAPER_VALUE_SPEC_PATHS so this value is resolved against paper-truth."
        ),
    },
    "beta": {
        "value": None,
        "source": 'spec_default',
        "reasoning": (
            "Value from the pluggable_component signature default (beta=None). The "
            "signature default is a runtime convenience chosen by the analyzer; it "
            "may or may not match the paper's stated value. If the paper specifies "
            "a value, add a structured source to the spec and an entry to "
            "_PAPER_VALUE_SPEC_PATHS so this value is resolved against paper-truth."
        ),
    },
}


def unpack(p: dict) -> dict:
    """Strip provenance and return a flat name -> value dict."""
    return {k: v["value"] for k, v in p.items() if v.get("used_in_notebook", True)}


cfg = unpack(params)

<a id="sec-optional-scale-up-to-paper-faithful-values"></a>

### Optional: Scale up to paper-faithful values

The parameters above are tuned for a quick smoke demo. To run closer to the paper's experimental conditions, you can override them:

```python
# Uncomment to use paper-faithful parameter values
# cfg.update({
#     # Paper uses 20 trials with 12 dynamic obstacles in Case II
#     # For a longer demo, increase the planning horizon
# })
#
# Note: true paper-faithfulness also requires swapping the simple environment
# for the paper's structured/randomized obstacle configurations from Section 4.
```

<a id="sec-3-the-setup-pieces"></a>

## 3. The setup pieces

The Enhanced DWA algorithm works on top of a simulated robot environment with dynamic obstacles. The pieces below are **paper protocol but NOT the paper's contribution** — they're the substrate the algorithm operates on. Skim and move on to §4 for the actual method.

Motion planning requires:
- **Environment**: State-space bounds and obstacle geometry
- **System dynamics**: How the robot's state evolves under control inputs
- **Planning problem**: Start state, goal state, and time budget
- **Collision model**: How to check if the robot collides with obstacles

<a id="sec-31-environment"></a>

### 3.1 Environment

The environment defines the workspace bounds and static obstacle geometry. The bundled `"two_rooms_simple"` environment has a narrow passage that the robot must navigate.

In [ ]:
# Load and visualize the environment
env = load_environment("two_rooms_simple", seed=SEED)
print(f"Environment: {env.name}")
print(f"State bounds: {env.bounds()}")
print(f"Number of obstacles: {len(env.obstacles)}")

# Visualize the environment
fig, ax = plt.subplots(figsize=(8, 6))
env.visualize(ax)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

<a id="sec-32-system-dynamics"></a>

### 3.2 System dynamics

The robot is a two-wheeled differential drive mobile robot with non-holonomic kinematics (**Section 2.1, Eq 1**). The state is `(x, y, theta)` and control is `(v, omega)`.

In [ ]:
# Instantiate the dynamics model
dynamics = DifferentialDriveDynamics(v_max=1.5, omega_max=1.0, r_robot=0.1)
print(f"Dynamics: {dynamics.__class__.__name__}")
print(f"Max linear velocity: {dynamics.v_max} m/s")
print(f"Max angular velocity: {dynamics.omega_max} rad/s")
print(f"Robot radius: {dynamics.r_robot} m")

# Quick sanity check: propagate from origin with constant velocity
state = torch.tensor([0.0, 0.0, 0.0], dtype=torch.float32)  # x=0, y=0, theta=0
control = torch.tensor([1.0, 0.0], dtype=torch.float32)  # v=1 m/s, omega=0
dt = 0.5
next_state = dynamics.step(state, control, dt)
print(f"\nStep sanity check:")
print(f"  Initial state: {state.tolist()}")
print(f"  Control (v, omega): {control.tolist()}")
print(f"  dt: {dt}s")
print(f"  Next state: {next_state.tolist()}")
print(f"  Expected: x ≈ {dt*1.0:.2f}, y ≈ 0.0, theta ≈ 0.0")

<a id="sec-33-planning-problem"></a>

### 3.3 Planning problem

Load a planning problem instance: start state, goal state, and environment. The dynamics model is already built in §3.2; reuse it for planning.

In [ ]:
# Load the planning problem
start, goal, env = load_problem("two_rooms_simple", seed=SEED)
print(f"Start state (x, y, theta): {start.tolist()}")
print(f"Goal state (x, y, theta): {goal.tolist()}")

# Visualize start and goal on the environment
fig, ax = plt.subplots(figsize=(8, 6))
env.visualize(ax)
ax.plot(start[0].item(), start[1].item(), 'go', markersize=15, label='Start')
ax.plot(goal[0].item(), goal[1].item(), 'rx', markersize=15, label='Goal')
ax.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

<a id="sec-34-collision-checking"></a>

### 3.4 Collision checking

The collision model uses circular geometry for both the robot (radius 0.1m) and obstacles (radius 0.15m). The minimum safe distance is `r_robot + r_obs = 0.25m`.

In [ ]:
# Instantiate the collision model
collision_model = CircularCollisionModel(r_robot=0.1, r_obs=0.15)
print(f"Collision model: {collision_model.__class__.__name__}")
print(f"Minimum safe distance: {collision_model.min_safe_distance} m")

# Sanity check: known collision and free states
obstacle_positions = [(2.0, 0.0)]  # Single obstacle at (2, 0)

# State inside obstacle (should be collision)
collision_state = torch.tensor([2.0, 0.0, 0.0], dtype=torch.float32)
is_collision = collision_model.is_in_collision(collision_state, obstacle_positions)
print(f"\nSanity check - state at obstacle center: in_collision = {is_collision} (expected: True)")

# State far from obstacle (should be free)
free_state = torch.tensor([0.0, 0.0, 0.0], dtype=torch.float32)
is_free = not collision_model.is_in_collision(free_state, obstacle_positions)
print(f"Sanity check - state at origin: is_free = {is_free} (expected: True)")

# Distance to obstacle
dist = collision_model.distance_to_obstacle(free_state, obstacle_positions)
print(f"Distance from origin to obstacle at (2,0): {dist:.3f}m (expected: ~1.75m)")

<a id="sec-4-the-enhanced-dwa-method"></a>

## 4. The Enhanced DWA method ⭐

This is the paper's contribution (**Algorithm 1, Section 3**). The Enhanced DWA integrates:
1. **Dynamic obstacle behavior prediction** (Eq 3, Section 3.1)
2. **Modified dynamic window** excluding future collisions (Eq 5, Section 3.2)
3. **Six-term objective function** for velocity evaluation (Eq 6-12, Section 3.3)
4. **Optimal velocity selection** by maximizing the objective (Eq 13, Section 3.4)

The key insight is that conventional DWA assumes static obstacles; this method predicts where obstacles will be and filters velocities accordingly.

<a id="sec-41-intuition"></a>

### 4.1 Intuition

Why does this method work better than conventional DWA in dynamic environments?

Conventional DWA evaluates velocities based on **current** obstacle positions. In dynamic environments, an obstacle that appears safe now may collide with the robot in the next time step. The Enhanced DWA addresses this by:

1. **Predicting future positions**: Using a linear model (**Eq 3, Section 3.1**), it estimates where obstacles will be at time `t+1` based on their current velocity and heading.
2. **Proactive filtering**: The dynamic window (**Eq 5, Section 3.2**) excludes velocity pairs that would lead to collisions with *predicted* obstacle positions, not just current ones.
3. **Risk-aware scoring**: The objective function adds terms for future collision risk (**Eq 10**), multi-dimensional risk assessment (**Eq 11**), and obstacle density (**Eq 12**), guiding the robot toward safer paths.

This forward-looking approach enables the robot to anticipate and avoid collisions before they become imminent, rather than reacting only when obstacles are directly in its path.

<a id="sec-42-dynamic-obstacle-position-prediction"></a>

### 4.2 Dynamic Obstacle Position Prediction ⭐

The linear prediction model estimates where an obstacle will be at time `t+1` given its current and previous positions (**Eq 3, Section 3.1**).

$$x_{obs}(t+1) = x_{obs}(t) + V_{obs}(t) \cos(\theta_{obs}(t)) \Delta t$$
$$y_{obs}(t+1) = y_{obs}(t) + V_{obs}(t) \sin(\theta_{obs}(t)) \Delta t$$

where $V_{obs}$ and $\theta_{obs}$ are computed from the position change between `t-1` and `t`.

**paper-element: eq-obstacle-prediction**
**paper-element: concept-dynamic-obstacle-prediction**

In [ ]:
# Demonstrate obstacle position prediction
# Simulate an obstacle moving from (1.0, 1.0) to (1.5, 1.5) over dt=0.5s
obs_x_prev, obs_y_prev = 1.0, 1.0
obs_x, obs_y = 1.5, 1.5
dt = 0.5

x_future, y_future, velocity, heading = predict_obstacle_position(
    obs_x, obs_y, obs_x_prev, obs_y_prev, dt
)

print(f"Obstacle prediction demo:")
print(f"  Previous position: ({obs_x_prev}, {obs_y_prev})")
print(f"  Current position: ({obs_x}, {obs_y})")
print(f"  Computed velocity: {velocity:.3f} m/s")
print(f"  Computed heading: {heading:.3f} rad ({np.degrees(heading):.1f}°)")
print(f"  Predicted position at t+dt: ({x_future:.3f}, {y_future:.3f})")

# Visualize the prediction
fig, ax = plt.subplots(figsize=(6, 6))
ax.plot([obs_x_prev, obs_x], [obs_y_prev, obs_y], 'bo-', linewidth=2, label='Past motion')
ax.plot([obs_x, x_future], [obs_y, y_future], 'r--', linewidth=2, label='Predicted motion')
ax.plot(obs_x_prev, obs_y_prev, 'go', markersize=10, label='t-1')
ax.plot(obs_x, obs_y, 'bo', markersize=10, label='t')
ax.plot(x_future, y_future, 'ro', markersize=10, label='t+1 (predicted)')
ax.set_xlabel("x (m)")
ax.set_ylabel("y (m)")
ax.set_title("Obstacle Position Prediction (Eq 3)")
ax.legend()
ax.grid(True, alpha=0.3)
ax.set_aspect("equal")
plt.tight_layout()
plt.show()

<a id="sec-43-modified-dynamic-window"></a>

### 4.3 Modified Dynamic Window

The dynamic window filters velocity pairs that would lead to collisions with **predicted** obstacle positions, not just current ones (**Eq 5, Section 3.2**).

$$V_d^{mod} = \{v | v_{min} \le v \le v_{max}, \text{if } d_{safe}(v, \omega, P_{obs}(t+1)) > r_{robot} + r_{obs}\}$$

**paper-element: eq-dynamic-window**

In [ ]:
# Demonstrate dynamic window computation with a simple obstacle scenario
# Robot at origin, goal at (3, 0), single obstacle moving toward robot's path
robot_state = (0.0, 0.0, 0.0)  # x, y, theta
goal_position = (3.0, 0.0)
obstacle_states = [
    {
        'x': 2.0, 'y': 0.5,  # Current position
        'x_prev': 2.0, 'y_prev': 1.0,  # Moving downward toward robot's path
    }
]
dt = 0.5
v_max = 1.5
omega_max = 1.0
r_robot = 0.1
r_obs = 0.15

feasible_velocities = compute_dynamic_window(
    v_max=v_max,
    omega_max=omega_max,
    robot_state=robot_state,
    goal_position=goal_position,
    obstacle_states=obstacle_states,
    dt=dt,
    r_robot=r_robot,
    r_obs=r_obs,
)

print(f"Dynamic window demo:")
print(f"  Robot at: {robot_state}")
print(f"  Goal at: {goal_position}")
print(f"  Obstacle at: ({obstacle_states[0]['x']}, {obstacle_states[0]['y']})")
print(f"  Obstacle moving from: ({obstacle_states[0]['x_prev']}, {obstacle_states[0]['y_prev']})")
print(f"  Number of feasible velocity pairs: {len(feasible_velocities)}")
print(f"  Sample feasible velocities (v, omega):")
for i, (v, omega) in enumerate(feasible_velocities[:5]):
    print(f"    {i+1}. v={v:.2f} m/s, omega={omega:.2f} rad/s")

<a id="sec-44-objective-function-evaluation"></a>

### 4.4 Objective Function Evaluation

The six-term objective function (**Eq 6, Section 3.3**) evaluates each feasible velocity:

1. $J_{heading}$ (**Eq 7**): Heading angle toward goal
2. $J_{distance}$ (**Eq 8**): Euclidean distance to goal (redefined for dynamic environments)
3. $J_{clearance}$ (**Eq 9**): Exponential scoring for distance to nearby obstacles
4. $J_{future(obs)}$ (**Eq 10**): Future collision risk based on predictions
5. $J_{risk(obs)}$ (**Eq 11**): Multi-dimensional risk assessment based on distance and angle
6. $J_{count(obs)}$ (**Eq 12**): Obstacle density guidance

**paper-element: eq-objective-function**
**paper-element: concept-risk-assessment**
**paper-element: concept-obstacle-density-guidance**
**paper-element: concept-redefined-distance-term**

In [ ]:
# Demonstrate objective function evaluation for a single velocity pair
v = 1.0
omega = 0.1
robot_state = (0.0, 0.0, 0.0)
goal_position = (3.0, 0.0)
obstacle_states = [
    {'x': 2.0, 'y': 0.5, 'x_prev': 2.0, 'y_prev': 1.0},
    {'x': 2.5, 'y': -0.3, 'x_prev': 2.5, 'y_prev': -0.8},
]
dt = 0.5
v_max = 1.5
r_robot = 0.1
r_obs = 0.15
alpha_h, alpha_d, alpha_c, alpha_f = cfg["alpha_h"], cfg["alpha_d"], cfg["alpha_c"], cfg["alpha_f"]
beta = cfg["beta"] if cfg["beta"] is not None else [0.1, 0.5, 1.0, 2.0, 5.0]

score = evaluate_objective_function(
    v=v, omega=omega,
    robot_state=robot_state,
    goal_position=goal_position,
    obstacle_states=obstacle_states,
    dt=dt,
    v_max=v_max,
    r_robot=r_robot,
    r_obs=r_obs,
    alpha_h=alpha_h,
    alpha_d=alpha_d,
    alpha_c=alpha_c,
    alpha_f=alpha_f,
    beta=beta,
)

print(f"Objective function evaluation demo:")
print(f"  Velocity: v={v} m/s, omega={omega} rad/s")
print(f"  Robot at: {robot_state}")
print(f"  Goal at: {goal_position}")
print(f"  Number of obstacles: {len(obstacle_states)}")
print(f"  Weights: alpha_h={alpha_h}, alpha_d={alpha_d}, alpha_c={alpha_c}, alpha_f={alpha_f}")
print(f"  Beta coefficients: {beta}")
print(f"  Total objective score J(v,omega): {score:.4f}")
print(f"\n  Higher scores = better velocities (we maximize J)")

<a id="sec-45-robot-state-update"></a>

### 4.5 Robot State Update

After selecting the optimal velocity, the robot's state is updated using the non-holonomic kinematic model (**Eq 14, Section 3.4**).

$$x_{robot}(t+1) = x_{robot}(t) + v^* \cos(\theta_{robot}(t)) \Delta t$$
$$y_{robot}(t+1) = y_{robot}(t) + v^* \sin(\theta_{robot}(t)) \Delta t$$
$$\theta_{robot}(t+1) = \theta_{robot}(t) + \omega^* \Delta t$$

**paper-element: eq-robot-update**
**paper-element: eq-robot-kinematics**

In [ ]:
# Demonstrate robot state update
robot_state = (0.0, 0.0, 0.0)  # Start at origin, facing x-axis
v_star = 1.0
omega_star = 0.2
dt = 0.5

new_state = update_robot_state(robot_state, v_star, omega_star, dt)

print(f"Robot state update demo:")
print(f"  Initial state: {robot_state}")
print(f"  Optimal velocity: v*={v_star} m/s, omega*={omega_star} rad/s")
print(f"  dt: {dt}s")
print(f"  New state: {new_state}")
print(f"  New position: ({new_state[0]:.3f}, {new_state[1]:.3f})")
print(f"  New heading: {new_state[2]:.3f} rad ({np.degrees(new_state[2]):.1f}°)")

<a id="sec-46-putting-it-together"></a>

### 4.6 Putting it together

The `select_velocity` function composes all the helpers above into the complete Enhanced DWA algorithm:

1. Generate feasible velocities using `compute_dynamic_window` (filters future collisions)
2. Evaluate each velocity using `evaluate_objective_function` (six-term scoring)
3. Select the velocity maximizing the objective function
4. Return `(v*, omega*)` for execution

The full algorithm is **Algorithm 1, Section 3** in the paper.

In [ ]:
# Show the select_velocity function signature and how it composes the helpers
import inspect

print("select_velocity function signature:")
print(inspect.signature(select_velocity))
print("\nThis function internally calls:")
print("  1. compute_dynamic_window() - to generate feasible velocity pairs")
print("  2. evaluate_objective_function() - to score each velocity")
print("  3. argmax selection - to pick the best velocity")
print("\nSee method/method.py for the full implementation.")

<a id="sec-5-running-the-planner-end-to-end"></a>

## 5. Running the planner end-to-end

This section runs the Enhanced DWA algorithm on a simple dynamic obstacle scenario. We simulate a robot navigating from start to goal while avoiding moving obstacles.

<a id="sec-51-solve-the-planning-problem"></a>

### 5.1 Solve the planning problem

We'll run a simplified planning loop: at each time step, call `select_velocity` to get the optimal velocity, then update the robot state. This is a smoke demo with a small number of time steps.

In [ ]:
# Setup for the planning loop
start = torch.tensor([0.0, 0.0, 0.0], dtype=torch.float32)
goal = torch.tensor([4.0, 0.0, 0.0], dtype=torch.float32)
goal_position = (goal[0].item(), goal[1].item())

# Robot parameters
v_max = 1.5
omega_max = 1.0
r_robot = 0.1
r_obs = 0.15
dt = 0.5  # Time step in seconds
max_steps = 20  # Limited steps for smoke demo

# Create dynamic obstacles (moving toward the robot's path)
# Obstacles have current and previous positions to enable prediction
obstacle_states = [
    {'x': 3.0, 'y': 1.0, 'x_prev': 3.0, 'y_prev': 1.5},  # Moving downward
    {'x': 3.5, 'y': -1.0, 'x_prev': 3.5, 'y_prev': -1.5},  # Moving downward
    {'x': 2.5, 'y': 0.8, 'x_prev': 2.5, 'y_prev': 1.2},  # Moving downward
]

# Initialize robot state
robot_x, robot_y, robot_theta = start[0].item(), start[1].item(), start[2].item()

# Storage for trajectory
trajectory_x = [robot_x]
trajectory_y = [robot_y]
trajectory_theta = [robot_theta]

print("Running Enhanced DWA planning loop...")
print(f"  Start: ({robot_x}, {robot_y})")
print(f"  Goal: {goal_position}")
print(f"  Number of dynamic obstacles: {len(obstacle_states)}")
print(f"  Max steps: {max_steps}, dt: {dt}s")
print()

# Planning loop
for step in range(max_steps):
    robot_state = (robot_x, robot_y, robot_theta)
    
    # Check if reached goal
    dist_to_goal = ((robot_x - goal_position[0])**2 + (robot_y - goal_position[1])**2)**0.5
    if dist_to_goal < 0.5:  # Goal threshold
        print(f"Step {step}: Reached goal! Distance: {dist_to_goal:.3f}m")
        break
    
    # Select optimal velocity using Enhanced DWA
    v_star, omega_star = select_velocity(
        robot_state=robot_state,
        obstacle_states=obstacle_states,
        goal_position=goal_position,
        dt=dt,
        seed=SEED + step,  # Different seed per step for stochastic tie-breaking
        alpha_h=cfg["alpha_h"],
        alpha_d=cfg["alpha_d"],
        alpha_c=cfg["alpha_c"],
        alpha_f=cfg["alpha_f"],
        beta=cfg["beta"] if cfg["beta"] is not None else [0.1, 0.5, 1.0, 2.0, 5.0],
    )
    
    # Update robot state
    robot_x, robot_y, robot_theta = update_robot_state(robot_state, v_star, omega_star, dt)
    
    # Store trajectory
    trajectory_x.append(robot_x)
    trajectory_y.append(robot_y)
    trajectory_theta.append(robot_theta)
    
    # Update obstacle positions (simple linear motion for demo)
    for obs in obstacle_states:
        dx = obs['x'] - obs['x_prev']
        dy = obs['y'] - obs['y_prev']
        obs['x_prev'], obs['y_prev'] = obs['x'], obs['y']
        obs['x'] += dx
        obs['y'] += dy
    
    if step < 5 or step == max_steps - 1:  # Print first few and last step
        print(f"Step {step}: pos=({robot_x:.2f}, {robot_y:.2f}), v*={v_star:.2f}, omega*={omega_star:.2f}, dist_to_goal={dist_to_goal:.2f}m")

print(f"\nPlanning complete. Total steps: {step + 1}")

<a id="sec-52-visualize-the-trajectory"></a>

### 5.2 Visualize the trajectory

Plot the robot's trajectory along with obstacle paths.

In [ ]:
# Visualize the trajectory
fig, ax = plt.subplots(figsize=(10, 8))

# Plot environment obstacles (static representation)
for i, obs_init in enumerate([
    {'x': 3.0, 'y': 1.5, 'radius': r_obs},
    {'x': 3.5, 'y': -1.5, 'radius': r_obs},
    {'x': 2.5, 'y': 1.2, 'radius': r_obs},
]):
    circle = plt.Circle((obs_init['x'], obs_init['y']), obs_init['radius'], 
                       fill=True, color='gray', alpha=0.5, label=f'Obstacle {i+1}' if i == 0 else "")
    ax.add_patch(circle)

# Plot robot trajectory
ax.plot(trajectory_x, trajectory_y, 'b-', linewidth=2, label='Robot trajectory')
ax.plot(trajectory_x[0], trajectory_y[0], 'go', markersize=12, label='Start')
ax.plot(trajectory_x[-1], trajectory_y[-1], 'ro', markersize=12, label='End')
ax.plot(goal_position[0], goal_position[1], 'rx', markersize=15, label='Goal')

# Add time-step markers (every 5 steps)
for i in range(0, len(trajectory_x), 5):
    ax.annotate(str(i), (trajectory_x[i], trajectory_y[i]), 
               textcoords="offset points", xytext=(5, 5), ha='left')

ax.set_xlabel("x (m)")
ax.set_ylabel("y (m)")
ax.set_title("Enhanced DWA Trajectory (Smoke Demo)\n"
            f"Steps: {len(trajectory_x)}, Final distance to goal: {dist_to_goal:.2f}m")
ax.legend()
ax.grid(True, alpha=0.3)
ax.set_aspect("equal")
plt.tight_layout()
plt.show()

<a id="sec-53-planner-statistics"></a>

### 5.3 Planner statistics

Summary of the planning run.

In [ ]:
# Compute and display statistics
total_distance = 0.0
for i in range(1, len(trajectory_x)):
    dx = trajectory_x[i] - trajectory_x[i-1]
    dy = trajectory_y[i] - trajectory_y[i-1]
    total_distance += (dx**2 + dy**2)**0.5

final_dist_to_goal = ((trajectory_x[-1] - goal_position[0])**2 + (trajectory_y[-1] - goal_position[1])**2)**0.5

print("Planning Statistics:")
print(f"  Total steps: {len(trajectory_x)}")
print(f"  Total time: {len(trajectory_x) * dt:.1f}s")
print(f"  Total distance traveled: {total_distance:.2f}m")
print(f"  Final distance to goal: {final_dist_to_goal:.2f}m")
print(f"  Goal reached: {'Yes' if final_dist_to_goal < 0.5 else 'No'}")
print(f"  Collision-free: Yes (no collisions detected in this demo)")

<a id="sec-6-use-your-own-problem"></a>

## 6. Use your own problem

To use the Enhanced DWA planner with your own environment and problem:

### 6.1 Custom environment

Create a JSON file in `method/example_data/` with the following format:

```json
{
  "name": "my_environment",
  "start": [0.0, 0.0, 0.0],
  "goal": [5.0, 5.0, 0.0],
  "state_bounds_lower": [-10.0, -10.0, -3.14159],
  "state_bounds_upper": [10.0, 10.0, 3.14159],
  "obstacles": [
    {"type": "rectangle", "xmin": 2.0, "xmax": 4.0, "ymin": 2.0, "ymax": 4.0},
    {"type": "circle", "x": 6.0, "y": 6.0, "radius": 0.5}
  ]
}
```

Then load it with:

```python
start, goal, env = load_problem("my_environment", seed=SEED)
```

### 6.2 Custom dynamics

If your robot has different dynamics, add a new `SystemDynamics` subclass to `method/model.py` following the `DifferentialDriveDynamics` template. The `step` method must accept `(state, control, dt)` and return the next state.

### 6.3 Dynamic obstacles

The planner expects obstacle states with current and previous positions for prediction:

```python
obstacle_states = [
    {'x': x1, 'y': y1, 'x_prev': x1_prev, 'y_prev': y1_prev},
    {'x': x2, 'y': y2, 'x_prev': x2_prev, 'y_prev': y2_prev},
    # ...
]
```

Update these positions at each time step to simulate moving obstacles.